# System Integration

This is the fifth notebook in the Elo7 Data Science Challenge project (full problem statement:
[`../docs/elo7-ds-challenge-en.md`](../docs/elo7-ds-challenge-en.md); notebooks 01-04 build the
EDA, product classifier, intent classifier, and recommender this notebook assembles).

## The task

Two things, both due here:

- **Part 5 (System Integration)**: a single system that takes an arbitrary search query and
  returns three outputs: the predicted product category (as if the query were a product), the
  predicted intent, and the ten most relevant recommended products.
- **Part 6 (Delivery)**: a command-line script exposing the whole solution, with three modes:
  `--category` (classify a real product's features), `--intent` (classify a query), and
  `--recommendation` (the full Part 5 system).

## The one open technical problem

Notebook 02's product classifier needs text **and** four numeric features. A bare search query
has no price, no weight, no minimum quantity, nothing but text. Notebook 02 deliberately left
this problem for here: "no numerical features are available; you should explore how to handle
this limitation," per the spec.

Two ways to handle it, compared directly in §2 rather than assumed:

1. Fit a **dedicated text-only classifier**, reusing the same TF-IDF vector space already
   shared across notebooks 02 and 04.
2. Reuse the existing text+numeric pipeline, but **impute the missing numeric features with
   their training-set median** at inference time.

## Questions this notebook answers

1. **Query -> category**: which of the two approaches above actually works better, and why?
2. **The pipeline**: how do the three subsystems (category, intent, recommendation) combine
   into one function?
3. **Generalization**: does the finished system actually work on queries that were never in the
   dataset at all?
4. **Delivery**: does the command-line script work exactly as specified, as a real subprocess,
   not just as Python function calls?

## 1. Setup

We reload every artifact notebooks 02-04 already produced: the product classifier (for its
fitted TF-IDF vectorizer, reused again here), the intent classifier, and the deduplicated
product catalog needed to rebuild the recommender's vector space.

In [2]:
import ast
import subprocess
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

from src.data.load import load_processed_dataset
from src.data.preprocess import deduplicate_products
from src.features.text import joined_tokens
from src.models.classifier import NUMERIC_FEATURES, TEXT_FEATURE, build_pipeline, build_query_category_pipeline
from src.models.recommender import build_catalog
from src.pipeline import run_pipeline

RANDOM_STATE = 42

df = load_processed_dataset("01_data.parquet")
products = deduplicate_products(df)
products[TEXT_FEATURE] = [
    joined_tokens(t, tg) for t, tg in zip(products["title"], products["concatenated_tags"])
]

classifier_pipeline = joblib.load("../models/02_product_classifier.joblib")
tfidf = classifier_pipeline.named_steps["preprocess"].named_transformers_["text"]
intent_pipeline = joblib.load("../models/03_intent_classifier.joblib")

print(f"{len(products):,} products loaded")

29,778 products loaded


## 2. The query -> category problem

We reproduce notebook 02's exact train/test split (`test_size=0.25`, stratified,
`random_state=42`) so the comparison below is apples to apples with what notebook 02 already
reported, and refit a fresh copy of the full pipeline (not the artifact notebook 02 saved, which
was already refit on the entire catalog and would leak test rows into this evaluation).

In [3]:
train, test = train_test_split(
    products, test_size=0.25, stratify=products["category"], random_state=RANDOM_STATE
)

full_pipeline_fresh = build_pipeline()
full_pipeline_fresh.fit(train[[TEXT_FEATURE, *NUMERIC_FEATURES]], train["category"])
pred_real_numeric = full_pipeline_fresh.predict(test[[TEXT_FEATURE, *NUMERIC_FEATURES]])

print(f"reference (text + real numeric features): macro-F1 = {f1_score(test['category'], pred_real_numeric, average='macro'):.3f}")

reference (text + real numeric features): macro-F1 = 0.849


That matches notebook 02's reported 0.849, confirming this split reproduces cleanly. Now the
two candidates for query time, when no numeric features exist at all:

In [4]:
# Approach 1: dedicated text-only classifier.
text_only_pipeline = build_query_category_pipeline()
text_only_pipeline.fit(train[TEXT_FEATURE], train["category"])
pred_text_only = text_only_pipeline.predict(test[TEXT_FEATURE])

# Approach 2: full pipeline, numeric features imputed with the training median.
median_values = train[NUMERIC_FEATURES].median()
test_imputed = test.copy()
for column in NUMERIC_FEATURES:
    test_imputed[column] = median_values[column]
pred_imputed = full_pipeline_fresh.predict(test_imputed[[TEXT_FEATURE, *NUMERIC_FEATURES]])

comparison = pd.DataFrame(
    {
        "text-only classifier": {
            "macro_f1": f1_score(test["category"], pred_text_only, average="macro"),
            "accuracy": accuracy_score(test["category"], pred_text_only),
        },
        "full pipeline, median-imputed numeric": {
            "macro_f1": f1_score(test["category"], pred_imputed, average="macro"),
            "accuracy": accuracy_score(test["category"], pred_imputed),
        },
    }
).T
comparison["agreement_with_other_approach"] = (pred_text_only == pred_imputed).mean()
comparison

,macro_f1,accuracy,agreement_with_other_approach
text-only classifier,0.843058,0.875756,0.987374
"full pipeline, median-imputed numeric",0.841582,0.874009,0.987374


Essentially identical, within noise of each other, and the two approaches agree on 98.7% of
individual predictions. That's not a coincidence: imputing every query with the *same constant*
numeric values contributes no discriminative signal at all (every example gets an identical
value for `price`, `weight`, and so on), so it's mathematically close to just dropping the
features, only with extra code and the conceptual awkwardness of pretending a query has a
price. **We use the dedicated text-only classifier** (`src.models.classifier.build_query_category_pipeline`):
simpler, equally accurate, and honest about what information is actually available at query
time. It costs 0.006 macro-F1 against the text+real-numeric upper bound, a small, explained
price for not fabricating numbers.

Refit on the full catalog before saving, matching the convention notebooks 02-03 already
established.

In [5]:
query_category_pipeline = build_query_category_pipeline()
query_category_pipeline.fit(products[TEXT_FEATURE], products["category"])

# Saved now, not deferred to the closing section: app/cli.py's --recommendation mode
# (demonstrated live in §5) loads this artifact and needs it to already exist on disk.
joblib.dump(query_category_pipeline, "../models/05_query_category_classifier.joblib")

for query in ["anel de prata", "tapete de croche", "dia dos pais"]:
    pred = query_category_pipeline.predict([joined_tokens(query)])[0]
    print(f"{query!r:30s} -> {pred}")

'anel de prata'                -> Bijuterias e Jóias
'tapete de croche'             -> Decoração
'dia dos pais'                 -> Lembrancinhas


## 3. Building the end-to-end pipeline

`src.pipeline.run_pipeline` ties the three subsystems together: the text-only category
classifier from §2, notebook 03's intent classifier, and notebook 04's hybrid recommender,
which already reuses the intent classifier internally to decide whether to diversify results.

In [6]:
catalog = build_catalog(products, tfidf)

result = run_pipeline("anel de prata", query_category_pipeline, intent_pipeline, catalog, k=10)
print("category:", result.category)
print("intent:", result.intent)
result.recommendations[["product_id", "title", "category"]]

category: Bijuterias e Jóias
intent: specific


,product_id,title,category
0,14308022,Anel Claddagh em Prata,Bijuterias e Jóias
1,12722782,Anel Trançado em Prata,Bijuterias e Jóias
2,235944,Anel em prata maleável,Bijuterias e Jóias
3,10202276,Anel de Prata Letras - Anel de Prata Feminino ...,Bijuterias e Jóias
4,10764366,Anel 4 Safiras em PRATA 925 - FRETE GRÁTIS,Bijuterias e Jóias
5,7766241,Anel de Prata De Letra Dupla - Prata 950 Anel ...,Bijuterias e Jóias
6,1064039,Anel em Prata de Coração,Bijuterias e Jóias
7,7815481,Anel Solitário de Prata 925 com Ródio,Bijuterias e Jóias
8,7865498,Anel Prata Escrava,Bijuterias e Jóias
9,16167852,Aliança de prata,Bijuterias e Jóias


All three outputs Part 5 asks for, from one query, one function call. A second example, a
broader query, to see the intent-aware diversification from notebook 04 show up inside the full
pipeline rather than in isolation:

In [7]:
result2 = run_pipeline("dia dos pais", query_category_pipeline, intent_pipeline, catalog, k=10)
print("category:", result2.category)
print("intent:", result2.intent)
result2.recommendations["category"].value_counts()

category: Lembrancinhas
intent: exploratory


category
Lembrancinhas    2
Outros           2
Papel e Cia      2
Decoração        2
Bebê             2
Name: count, dtype: int64

## 4. Testing with unseen queries

Every evaluation in notebooks 02-04 ultimately replayed historical data in some form: held-out
rows of the same dataset, or clicks that already happened. Part 5's actual requirement is
broader: **any** query, including ones this dataset has never seen. We test with genuinely novel
phrasings, invented for this notebook, not lightly-reworded historical queries.

In [8]:
unseen_queries = [
    "presente de aniversario para amiga",
    "colar masculino aco inoxidavel",
    "kit maternidade completo enxoval",
    "convite digital formatura",
]

for query in unseen_queries:
    in_dataset = query in set(df["query"])
    result = run_pipeline(query, query_category_pipeline, intent_pipeline, catalog, k=5)
    print(f"\n'{query}'  (seen historically: {in_dataset})")
    print(f"  category: {result.category}   intent: {result.intent}")
    print(result.recommendations[["title", "category"]].to_string(index=False))


'presente de aniversario para amiga'  (seen historically: False)
  category: Lembrancinhas   intent: exploratory
                                                 title      category
                                        ALMOFADA AMIGA     Decoração
Presente amiga minha alegria personalizado c/ sua foto Lembrancinhas
                                   Caixa para presente   Papel e Cia
                     Caixa para Presente Personalizada Lembrancinhas
                                      Presente 15 anos     Decoração

'colar masculino aco inoxidavel'  (seen historically: False)
  category: Bijuterias e Jóias   intent: specific
                                                      title           category
   Anel masculino aço inoxidável 316l carbono preto e prata Bijuterias e Jóias
                                    Colar Masculino Mandala Bijuterias e Jóias
       Colar masculino palheta guitarra placa personalizada Bijuterias e Jóias
Colar Aromatizador - Difusor Pessoal pequeno

None of these four exact strings appear anywhere in the historical dataset, and every one
produces a plausible category, a plausible intent, and recommendations that visibly relate to
what was asked, direct evidence the system generalizes rather than only memorizing patterns
from queries it was indirectly trained to recognize.

## 5. The command-line delivery script

`app/cli.py` implements the Part 6 contract: `--category` (a real product's features, the
ordinary Part 2 problem), `--intent` (a query), and `--recommendation` (the full pipeline from
§3-4). We run it as an actual subprocess for each mode, the real, literal delivery format the
spec asks for, not just a call into the underlying Python functions.

In [9]:
def run_cli(*args):
    result = subprocess.run(
        [sys.executable, "../app/cli.py", *args],
        capture_output=True,
        text=True,
        cwd=Path.cwd(),
    )
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result.stdout.strip()

**`--category`**, a real product description:

In [10]:
features = {
    "title": "Anel de Prata Feminino",
    "concatenated_tags": "anel prata feminino joia",
    "price": 49.9,
    "minimum_quantity": 1,
    "weight": 5,
}
print(run_cli("--category", str(features)))

Bijuterias e Jóias


**`--intent`**, a bare query:

In [11]:
print(run_cli("--intent", "dia dos pais"))

exploratory


**`--recommendation`**, the full hybrid system, exactly the spec's three-part output format
(category, then intent, then ten `product_id,title` lines):

In [12]:
print(run_cli("--recommendation", "anel de prata"))

Bijuterias e Jóias
specific
14308022,Anel Claddagh em Prata
12722782,Anel Trançado em Prata
235944,Anel em prata maleável
10202276,Anel de Prata Letras - Anel de Prata Feminino Letra L
10764366,Anel 4 Safiras em PRATA 925 - FRETE GRÁTIS
7766241,Anel de Prata De Letra Dupla - Prata 950 Anel Feminino
1064039,Anel em Prata de Coração
7815481,Anel Solitário de Prata 925 com Ródio
7865498,Anel Prata Escrava
16167852,Aliança de prata


All three modes run as genuinely separate processes, invoking the delivered script exactly as a
grader would, and produce output in the format the spec lays out.

## 6. Key findings & handoff

**The query -> category problem, resolved**: a dedicated text-only classifier, reusing the
shared TF-IDF vector space from notebooks 02 and 04, performs statistically indistinguishably
from imputing numeric features with their training median (macro-F1 0.843 vs. 0.842, 98.7%
prediction agreement), for a clear reason: a constant imputed value carries no discriminative
signal. The simpler, more honest option was kept.

**What was built:**

- `src/models/classifier.py`: `build_query_category_pipeline`, the text-only classifier from
  §2, persisted as `models/05_query_category_classifier.joblib`.
- `src/pipeline.py`: `run_pipeline`, the actual Part 5 hybrid system as importable code, so
  notebook 06 and `app/cli.py` both call the same function rather than duplicating logic.
- `app/cli.py`: the Part 6 delivery script, all three modes demonstrated as real subprocesses
  in §5.

**What we learned:**

- The query -> category gap costs very little (0.006 macro-F1) once the right substitute is
  used; the naive "impute and reuse" approach costs almost exactly the same amount for a worse
  reason (fabricated numbers instead of an honest text-only model).
- The finished system generalizes to genuinely novel queries (§4), not just held-out rows of
  data it was adjacent to during training.

**Handoff to notebook 06**: formal evaluation of all three subsystems together, including this
notebook's query-category classifier as one more component to score, using the metrics
established across notebooks 02-04 (macro-F1 for both classifiers, hit-rate/recall for the
recommender) rather than inventing new ones.

In [13]:
data_dir = Path("../data/processed")

feature_catalog = [
    ("query_category", "categorical", "engineered", "product_text (query)", "Predicted category for a query, via the text-only classifier."),
    ("query_intent", "categorical", "engineered", "product_text (query)", "Predicted intent for a query, via notebook 03's classifier."),
]
df_features = pd.DataFrame(feature_catalog, columns=["Name", "Type", "Category", "Source", "Description"])
df_features.to_parquet(data_dir / "05_features.parquet", index=False)

print(f"saved {len(df_features)} feature rows -> data/processed/05_features.parquet")

saved 2 feature rows -> data/processed/05_features.parquet


## Closing remarks

This notebook closed the one gap left open since notebook 02: classifying a query as if it were
a product, with no numeric features available, resolved with a dedicated text-only classifier
rather than fabricated numeric inputs, at a small, explained cost. The three subsystems built
across notebooks 02-04 now run as a single function (`src.pipeline.run_pipeline`) and a real
command-line script (`app/cli.py`), both demonstrated on queries the dataset has never seen.

The [next notebook](06_evaluation.ipynb) formally evaluates all three subsystems together.